# PROJECT REGENESIS
# TRACK A OPERATING MANUAL
## EEG Signal Processing & Decoding

**Canonical identity:** Track A = Track 1 of the Phase II Handbook §13, "EEG Signal Processing & Decoding."

**Status:** Standalone working manual for the Track A team. Everything required to execute this track independently is contained here. You should not need to open the Handbook during ordinary work; if you find yourself needing to, that is a defect in this manual and should be reported.

**Authority:** Derived entirely from the Phase II Handbook and the Phase II Execution Guide. Where this manual and the Handbook appear to disagree, **the Handbook governs** and you raise the discrepancy the same day. No statement in this manual is new science.

**Team model:** Track A operates as one collaborative research team. There is no internal ownership split. Every result carries the track's name; every result must be reproducible by any member of the track and by at least one member outside it.

---

# 0. STANDING RULES — READ ONCE, APPLY ALWAYS

These are laboratory-wide and are inlined here so you never have to leave this document.

### 0.1 Non-negotiables

| Rule | Consequence of breaking it |
|---|---|
| Splits are **block-wise** — by trial block, by session, or by subject. **Never window-level.** | Inflated accuracy by tens of points; result withdrawn |
| Splitters, surrogates, estimators and metrics come from `eval/` only. **No local copies.** | Results across tracks become non-comparable |
| Preprocessing statistics are fitted **within fold**. | Leakage; result withdrawn |
| Hyperparameters are selected on a **nested inner fold**, never the evaluation fold. | Slow leak; result withdrawn |
| Every experiment ships with its negative control, **named in the pre-registration before data is touched**. | Pre-registration not signed |
| Every claim carries its scope: population, condition, offline/online. | Ledger entry rejected |
| Every citation is opened, graded A–F, entered in the Evidence Table before use. **Unopened citations are deleted.** | Claim flagged unsupported |
| Reading is predict-then-read: expected method, expected result, one expected limitation, written **before** opening the paper. | Journal Club slot forfeited |

### 0.2 Evidence grades

**A** peer-reviewed, adequate design, target-adjacent population · **B** peer-reviewed, limiting design (single subject, offline only, non-target population, small *n*) · **C** preprint · **D** non-peer-reviewed (vendor docs, industry reports, blogs) · **F** unverifiable or quote not located — **deleted from the record**.

A quote from a paper's **Introduction is secondhand** — the paper repeating someone else, usually with conditions stripped. Grade one level down and trace to origin or remove.

### 0.3 Definition of done

A task is complete when **all** of the following exist:

- `results/PR-YYYY-NN/` containing `figure.png`, `claim.md` (**one sentence, with scope, nothing else**), `methods.md`, `surrogates.md`, `env.lock`, `data.sha256`, `seed.txt`, `deviations.md`.
- Someone outside Track A regenerated it from the repository in under thirty minutes.
- The relevant `ledger/` entry is updated or a new one opened.

Anything else is in progress, however finished it feels.

### 0.4 Failure and refutation are different fields

A **Refutation Condition** is defined on the hypothesis: the observation showing the claim is false.
A **Failure Condition** is defined on the project: the observation showing this path is blocked.

Both appear in every pre-registration, separately. We are indifferent between hypothesis outcomes and very much not indifferent between project outcomes.

### 0.5 The three standing questions

Asked of anything, by anyone, of anyone:

> **What is your control? What is your split? What would this look like if you were wrong?**

### 0.6 Escalate the same day if

You cannot reproduce a published number · your accuracy exceeds expectation by a wide margin (**suspect leakage before celebrating**) · dataset metadata disagrees with its publication · two Regenesis documents contradict each other · you find yourself constructing an argument for a result you have already decided is true.

---

# 1. TRACK OVERVIEW

## 1.1 Why this track exists

Regenesis rests on a single anatomical claim: for a transradial amputee, the muscles that would encode digit intent are gone, but the cortical machinery that planned their movement is intact. If that claim is to do any work, the surviving cortical activity must be **readable through a skull, at a usable rate, and distinguishable from muscle.** Track A determines whether it is.

The track therefore sets the upper bound on the entire programme. No amount of downstream fusion, representation learning or control sophistication can recover information that never survived the scalp. When Track A reports a number, every other track's ambition is bounded by it.

## 1.2 Which hypotheses this track supports

| Hypothesis | Track A's contribution |
|---|---|
| **H1** — EEG carries grasp information not redundant with simultaneously recorded EMG, largest under EMG-degraded conditions | Supplies the EEG features **and**, decisively, the controls that determine whether any positive result is cortical or myogenic |
| **H2** — a decodable fraction of EEG grasp/intent information is available before EMG onset at usable lead times and false-positive rates | Builds and sweeps the anticipatory detector |

Track A does **not** own H1 or H2. Track C (Fusion) owns H1 through E2 and the forward-feed half of H2. Track A owns the EEG-side inputs to both, and owns the artifact analysis outright.

## 1.3 The scientific question

> **What does scalp EEG actually deliver for grasp intent — at what class cardinality, with which features, under artifact-controlled and leakage-safe evaluation — and how much of any apparent signal is cortical rather than myogenic?**

## 1.4 Why the result matters

Three reasons, in ascending order of consequence.

**It selects the feature family.** The project originally specified mu/beta band power. The current architecture specifies low-frequency MRCP (0.3–3 Hz), because that is what the reach-and-grasp literature actually uses. Track A quantifies the difference on our data rather than inheriting the assertion.

**It bounds the architecture.** EEG is a discrete, low-rate source in the current architecture — never a continuous finger controller. That decision was made on an information ceiling of roughly r ≈ 0.3–0.5 for continuous kinematic decoding. Track A produces our own measurement of what class cardinality is reliably supported, which is what a grasp vocabulary can be built on.

**It decides whether Gate B is real.** This is the one that matters. Myogenic contamination of scalp EEG — cranial, facial, jaw and neck musculature — reproduces H1's *entire* confirmatory signature with no cortical contribution whatsoever. Track A owns the three controls that separate the two. If Gate B passes on artifact, every subsequent tier, every subsequent paper and the flagship are built on it.

## 1.5 Why the rest of the laboratory depends on this work

| Who | What they need from Track A | When |
|---|---|---|
| **Track C (Fusion)** | Band-resolved EEG features; montage-ablated variants; the myogenic component estimator; a jointly signed E2 pre-registration | Week 8 |
| **Track C (Fusion)** | The E3 detector output, for forward-feed and the latency-reduction measurement | Week 11 |
| **Track C (Fusion)** | An EEG encoder for E4's paired arm | Week 13+ |
| **Track E (Evidence)** | Our reproduced values and their deltas, for the Number Board | Week 4 |
| **Whole laboratory** | The honest statement of what EEG delivers, with its scope clause, entered in the belief ledger | Week 12 |

Track A is not on the critical path in Weeks 1–3 — Track E is — but from Week 8 onward **Track C cannot run E2 without Track A's controls.** Slipping the artifact work slips Gate B, and Gate B gates Tiers 2 and 3.

---

# 2. SCIENTIFIC BACKGROUND

Only what this team must understand completely before starting. Everything here is examinable without warning.

## 2.1 Where EEG sits in the project's ontology

Regenesis holds three levels rigorously separate:

- **Level 1 — motor intent `x(t)`.** The time-evolving, modality-independent internal state of the user's motor system. **Unobservable in principle.**
- **Level 2 — physiological observations `y_m(t)` = `g_m(x(t))` + noise.** What instruments measure. EEG is `g_EEG`; EMG is `g_EMG`. **No single projection is invertible.**
- **Level 3 — learned representations `z(t)`.** Our computational approximations of Level 1, estimated from Level 2. Always partial. **Never to be mistaken for `x(t)`.**

Two consequences you will use daily:

**Amputation is the loss of an observation channel, not of the intent.** A transradial amputation degrades or removes `g_EMG`; `x(t)` is untouched. This is the entire principled reason EEG is in the project — not "EEG is earlier," but "EEG is the remaining channel for a specific class of information."

**Temporal precedence is structural, not anecdotal.** `g_EEG` taps the state upstream of `g_EMG` in the projection chain, so EEG *can* carry `x(t)` before EMG does. That is the correct statement of the anticipation argument, and it is what E3 tests. The older readiness-potential framing — cortical potentials precede movement by 1–2 s, therefore we gain a head start — is retired, because the raw fact of precedence says nothing about whether the lead survives processing latency.

## 2.2 The physiology you must be able to explain on demand

**Event-related desynchronisation (ERD) and synchronisation (ERS).** Movement and motor imagery produce a decrease in mu (8–12 Hz) and beta (13–30 Hz) band power over sensorimotor cortex, followed by a post-movement rebound. This is the phenomenon the entire EEG path rests on. You must be able to state its approximate time course, its spatial distribution, and its inter-subject variability.

**Movement-related cortical potentials (MRCP).** Low-frequency (≈0.3–3 Hz) negative-going potentials preceding and accompanying voluntary movement. **This is our primary feature family**, adopted over band power because it is what the reach-and-grasp decoding literature uses. Band power is retained as a comparison arm.

**Volume conduction.** Cortical current sources are spatially blurred by the skull and scalp before reaching the electrode. Consequences: adjacent cortical representations mix; finger representations, which are adjacent and overlapping, are especially hard to separate; and **no downstream algorithm can undo this** — it is destruction, not merely mixing, at the resolution we care about.

**What a transradial amputation removes.** The forearm distal to the amputation level, including the intrinsic hand muscles and the distal portions of the extrinsic finger flexors and extensors. What remains: proximal forearm musculature, the elbow, and the entire central nervous system. This is why EMG can carry wrist and forearm intent and cannot carry digit intent.

**BCI illiteracy.** Roughly 15–30% of users do not produce classifiable motor-imagery patterns with usable reliability. This is a hard constraint, replicated across independent lines. It means system performance is partly a property of the user, not of the algorithm — and it is why every EEG-dependent path in the architecture has an EMG-only fallback.

## 2.3 The myogenic contamination mechanism — the most important thing in this section

You must be able to derive this from memory. It is the reason Track A's controls exist.

1. Scalp EEG contains myogenic activity from cranial, facial, jaw and neck musculature. This activity is **correlated with movement execution and with effort.**
2. That contamination therefore carries grasp-correlated information — not because cortex encoded it, but because **the body moved**.
3. Conditional mutual information `I(grasp ; EEG | EMG) > 0` follows immediately: the contamination is not redundant with *forearm* EMG, because it originates in entirely different muscles.
4. Now degrade the forearm EMG channel — attenuate it, add noise, simulate fatigue or dropout. **The contamination in the EEG channel is untouched.** Forearm-EMG information falls; EEG-side information holds constant; the fusion gain rises.
5. The gain-versus-degradation curve slopes upward **exactly as H1 predicts**.

**A label-shuffle permutation null does not catch this.** Shuffling grasp labels destroys the artifact's information along with the neural information, so the artifact-driven result passes the null comfortably. A null that both hypotheses pass is not a test.

Three properties of the contamination give us leverage, and they are the basis of our three controls:

- **Frequency.** Myogenic activity is broadband and dominant above roughly 20 Hz. MRCP features at 0.3–3 Hz are comparatively protected; mu/beta features are not.
- **Space.** Contamination is largest at peripheral, temporal and frontal sites. Genuine grasp information should be lateralised over contralateral sensorimotor cortex.
- **Separability.** A high-frequency myogenic component can be estimated from the EEG montage itself and conditioned on.

## 2.4 Evaluation concepts you must hold

**Leakage.** Any path by which information from the evaluation set influences training. In neural time-series the dominant forms are: overlapping windows from one trial placed in both train and test; trial-adjacent splitting; preprocessing statistics fitted across the full dataset; hyperparameter selection on the evaluation fold; and subject-pooled cross-validation reported as cross-subject. **These compound.** In this data family, evaluation policy is frequently a larger effect than method choice.

**Surrogate.** A transformation of the data that destroys the effect of interest while preserving as much else as possible. Ours: label shuffle, phase randomisation, spectrally matched noise, time reversal, channel permutation. Provided by `eval/surrogates/`, validated by Track E in **both** directions — a planted real effect must survive, and a planted artifact must die. A suite validated only against under-correction is not validated.

**Scope clause.** Every result carries its population, condition and offline/online status as part of the claim. Not *"EEG decodes grasp at 68%"* but *"EEG decodes 4-class grasp at 68% (per-subject median, 95% CI ...) in 12 able-bodied subjects, offline, session-wise splits, MRCP features, central-only montage."*

## 2.5 What is settled and what is open

**Settled enough to build on** — but every value is being re-verified by Track E's Numbers Audit before any threshold is set against it:
- EEG grasp decoding is discrete and modest: binary grasp-versus-grasp around the low 70s %, four-class peaking in the mid 60s %.
- Those results use low-frequency MRCP features, not mu/beta band power.
- Continuous kinematic/force decoding from scalp EEG has an honestly validated ceiling around r ≈ 0.3–0.5; several published values above 0.7 are likely inflated by low-frequency motion, EMG or mechanical artifact.
- BCI illiteracy affects 15–30% of users.

**Open, and what this track exists to address:**
- The magnitude and conditionality of unique EEG information beyond EMG (Track C owns the measurement; we own its validity).
- The real anticipatory operating curve for grasp *type*, as opposed to grasp *onset* — onset is comparatively well studied, type is not.
- How much of any apparent EEG contribution is myogenic.

---

# 3. SCOPE

## 3.1 Track A IS responsible for

- All EEG preprocessing: filtering, surface Laplacian, ICA and artifact handling, epoching.
- MRCP and mu/beta band-power feature extraction.
- Three decoder families — linear/sLDA, Riemannian tangent-space, EEGNet — implemented so that **method effect can be separated from signal effect**.
- The reproduction of a published motor-imagery decoding number under strict splits, with the delta reported.
- The grasp/onset decoding ladder with its full surrogate panel.
- **The three artifact controls of the E2 protocol: band-resolved features, montage ablation, and the myogenic component estimator.**
- The E3 anticipatory detector and its lead × TPR × FP/min sweep.
- Dataset cards for every EEG corpus used.
- An EEG encoder for E4's paired arm (Weeks 13+).
- Ledger maintenance for the EEG-related beliefs.

## 3.2 Track A is NOT responsible for

| Not ours | Whose | Why the boundary is here |
|---|---|---|
| Computing conditional mutual information | Track C | Track C owns H1 and the estimator; we supply the features and controls it conditions on |
| The fusion ladder and its tuning budgets | Track C | Single owner prevents divergent evaluation |
| Deciding Gate B | Track C | We supply the evidence that makes the decision valid |
| The forward-feed latency measurement in E3 | Track C | We build the detector; Track C measures what it buys downstream |
| Any EMG processing or synergy extraction | Track B | Including the EMG side of WAY-EEG-GAL |
| Splitters, surrogates, metrics, statistical utilities | Track E | Single implementation, laboratory-wide |
| Verifying published literature values for the Number Board | Track E | We supply our reproduced values; Track E audits the published ones |
| Any closed-loop or hardware work | Track D / Phase 2 | Behind Gate E |
| The band-power-NMF generative-justification simulation | **Explicitly deferred** | Open question Q7 in the Handbook; off the critical path; the mechanism it interrogates is already removed from the architecture. **Do not start it in Phase II unless the PI reassigns it in writing.** |

## 3.3 Boundary conditions

- **No EEG result of ours is ever presented as evidence about fusion.** We report what the EEG channel carries. Whether it carries anything *beyond* EMG is Track C's measurement.
- **No benchmark motor-imagery number is ever cited as bearing on H1.** BCI-IV-2a establishes that our pipeline works. It says nothing about grasp intent in this project.
- **Epoching is relative to EMG threshold-onset, not to trial start or cue.** Cue-aligned epoching silently converts an anticipation result into a cue-response result.
- **We do not modify Track B's EMG outputs**, and Track B does not modify ours.
- Where the data does not support a claim about grasp-type diversity, we do not make one. WAY-EEG-GAL has a narrow grasp vocabulary; this constraint is stated in every figure caption, not absorbed.

## 3.4 Deliverables expected from this track

Summarised here; specified in full in §12.

**Research:** the statement of what EEG delivers, per subject, with CIs and scope clauses; the MRCP-versus-band-power comparison quantified; the artifact-versus-cortical separation; the E3 operating curve.
**Software:** the MNE pipeline; feature extractors; three decoder families; band-resolved routing and montage-ablation utilities; the myogenic estimator.
**Documentation:** dataset cards; Number Board entries for our own values; pre-registrations; ledger updates.
**Presentation:** weekly Figure Clinic figure; Lab Meeting deep presentations on rotation; three Journal Club papers with reproduction attempts.
**Decision input:** the evidence packet on which Gate B's validity rests, and the E3 curve for Gate C.

---

# 4. RESEARCH QUESTIONS

Every question Track A is expected to answer. Confidence levels are the laboratory's current position and are ours to move.

### RQ-A1 — What is the reliable class cardinality for grasp decoding from scalp EEG in our data?

**Why it matters.** It bounds the grasp vocabulary the architecture can support. The current architecture treats EEG as a discrete low-rate selector; the value of *k* determines whether that selector is useful or trivial.
**Current belief.** Discrete and modest — binary around the low 70s %, four-class peaking in the mid 60s %.
**Confidence.** Medium, pending the Numbers Audit.
**Experiment.** T1-R (reproduction), then the decoding ladder within E2's EEG arm.
**Success criteria.** Per-subject accuracy at *k* = 2, 3, 4 reported with bootstrap CIs, on session-wise splits, surviving the full surrogate panel, with a stated scope clause.
**Failure criteria.** Cannot exceed the surrogate null at any *k* → report it. This is a Tier-1-relevant negative and it is publishable.

### RQ-A2 — Do MRCP features outperform mu/beta band power for grasp intent on our data?

**Why it matters.** The architecture already specifies MRCP over band power, on the strength of the published reach-and-grasp literature rather than our own measurement. This question converts an inherited assertion into a result.
**Current belief.** MRCP is the working feature family for grasp decoding; band power is not.
**Confidence.** Medium-High, pending the Numbers Audit.
**Experiment.** The MRCP-versus-band-power arm of the decoding ladder, with equal tuning budgets.
**Success criteria.** A quantified per-subject comparison with a paired statistical test and CIs.
**Failure criteria.** None — either outcome is informative. If band power wins, that is a finding and the ledger moves.

### RQ-A3 — How much of any apparent EEG grasp information is myogenic rather than cortical?

**Why it matters.** **This is the most consequential question in the track.** It determines whether Gate B's decision is valid, and therefore whether Tiers 2 and 3 are built on something real.
**Current belief.** The mechanism by which contamination reproduces H1's signature is established deductively; its magnitude in our data is unknown.
**Confidence.** High on the mechanism, Unknown on the magnitude.
**Experiment.** The three controls: band-resolved decoding, montage ablation, myogenic nuisance conditioning.
**Success criteria.** All three figures produced; the before/after myogenic-conditioning difference quantified and delivered to Track C.
**Failure criteria.** If the effect is concentrated above 20 Hz, collapses under a central-only montage, or largely disappears after myogenic conditioning — **that is the finding, it is reported as the headline, and Gate B is decided accordingly.** This is not a failure of the track.

### RQ-A4 — What is the anticipatory operating curve for grasp onset, and where the data permits, grasp type?

**Why it matters.** Anticipation is the one thing better EMG can never replace, because before onset there is no EMG signal to be redundant with. Grasp-*type* anticipation is substantially under-studied relative to onset, and the achievable lead-versus-false-positive trade-off is the quantity a designer actually needs.
**Current belief.** Usable lead at acceptable false-positive rates is plausible for onset; type anticipation is harder.
**Confidence.** Medium for onset, Low for type.
**Experiment.** E3 detector sweep.
**Success criteria.** The full lead × TPR × FP/min curve with per-subject spread — **not a single operating point**.
**Failure criteria.** Detection achievable only at unusable false-positive rates → report the curve as the finding. The curve is publishable whether or not the operating point is usable.

### RQ-A5 — Does subject variability exceed method variability in our EEG results?

**Why it matters.** It determines whether effort should go into better decoders or into screening and per-subject calibration, and it bears directly on the BCI-illiteracy constraint.
**Current belief.** BCI illiteracy at 15–30% is a hard constraint; between-subject variance is expected to be large.
**Confidence.** High that illiteracy is real; unmeasured in our data.
**Experiment.** Reported as a by-product of the decoding ladder — per-subject distributions across all three decoder families.
**Success criteria.** Per-subject × per-method table with variance attributable to each, reported alongside the ladder.
**Failure criteria.** None. This is descriptive and always informative.

---

# 5. EXPERIMENTS

Three experiments. No ambiguity is left; where a choice exists it is made here and any deviation is logged in `deviations.md`.

---

## 5.1 T1-R — Reproduction of a published motor-imagery decoding result

**Objective.** Establish that our pipeline works and that we can hit a published number under honest splits.

**Scientific motivation.** Before this track's own numbers mean anything, we must show we can reproduce someone else's under our evaluation policy. The delta between our value and the published one is itself informative: it calibrates how to read every EEG number we will later compare ourselves against, and it feeds Track E's Numbers Audit.

**Inputs.** BCI-IV-2a. One published two-class motor-imagery accuracy with its conditions extracted first: *n*, subjects, class definitions, chance level, split policy, feature family, classifier.

**Outputs.** Reproduced accuracy per subject; the delta from published; a written hypothesis for the discrepancy.

**Pipeline.**
1. Load through `data/` with a version hash recorded.
2. Band-pass 8–30 Hz. Epoch per the source paper's timing, recorded explicitly.
3. CSP or Riemannian tangent-space features, matching the source paper's family.
4. sLDA classifier.
5. **Block-wise split from `eval/splits/`** — session-wise where the dataset provides sessions, else trial-block.
6. Label-shuffle surrogate.
7. Per-subject accuracy, bootstrap CIs.

**Expected intermediate results.** After step 3, feature dimensionality and class separability should be visually inspectable — plot the first two CSP or tangent-space components per class. If classes are not visibly separated for the best subjects, stop and debug before proceeding to step 4.

**Controls.** Block-wise split; label-shuffle surrogate. Both mandatory.

**Failure modes.** Cannot locate the source paper's conditions → escalate, and grade the source down. Reproduce far above the published value → **suspect leakage before celebrating**; check the split first. Reproduce far below → check epoch timing and band definition before concluding anything.

**Evaluation metrics.** Per-subject accuracy; the delta from published; the surrogate null distribution.

**Statistical tests.** Bootstrap percentile CIs on per-subject accuracy (≥1000 resamples). Permutation test against the label-shuffle null. No paired test is required here; this is a single-arm reproduction.

**Expected figures.** *Figure T1-R.1:* per-subject reproduced-versus-published scatter with the identity line and CIs.

**Interpretation.** Points near the identity line: pipeline validated. Points systematically below: our evaluation is stricter, and the size of the gap is the calibration we needed. Points above: investigate leakage before anything else.

**Decision gate.** None directly. Blocks nothing formally, but no Track A modelling result is reported before T1-R closes.

**Publication relevance.** Feeds Track E's Numbers Audit and Publication 1. Not a standalone contribution.

**Estimated duration.** 2 weeks.

**Complete when.** Result directory complete per §0.3; delta reported with a hypothesis; Number Board entry created.

---

## 5.2 E2-A — Track A's contribution to the Tier 1 information decomposition

**Objective.** Supply Track C with EEG features and — decisively — with the evidence that separates cortical from myogenic contribution.

**Scientific motivation.** E2 tests H1. Its confirmatory signature is `I(grasp ; EEG | EMG) > 0` together with fusion gain growing under EMG degradation. That signature is reproducible in full by myogenic contamination with no cortical contribution (§2.3). Track A's controls are what make Gate B's decision mean anything. **This is the track's principal scientific responsibility.**

**Inputs.** WAY-EEG-GAL. Full and central-only montages. MRCP (0.3–3 Hz) and mu/beta (8–30 Hz) feature sets.

**Outputs, delivered to Track C.**
1. Band-resolved feature sets, so conditional MI can be reported per frequency band.
2. Montage-ablated variants: full montage versus central-only, with peripheral, temporal and frontal channels removed.
3. A high-frequency myogenic component estimated from the EEG montage itself, for use as a nuisance covariate.
4. A jointly signed E2 pre-registration.

**Pipeline.**
1. Load WAY-EEG-GAL; version hash recorded; **synchronisation measured, not assumed**.
2. Epoch relative to **EMG threshold-onset**, keeping pre-onset and post-onset windows separate throughout. Hand-verify onset labels on ≥30 trials before proceeding.
3. Two parallel feature paths: MRCP band-pass 0.3–3 Hz; band power 8–30 Hz.
4. Surface Laplacian. ICA for ocular artifact — **validated against a planted effect before adoption** (see failure modes).
5. Band-resolved routing: features computed and retained per band so per-band conditional MI is possible downstream.
6. Montage variants generated: full, central-only.
7. Myogenic component estimated from high-frequency content across the montage.
8. Decoding diagnostics per band and per montage, as our own read on what the controls will show Track C.

**Expected intermediate results.** After step 5, per-band decoding accuracy should be inspectable. Expect: MRCP band carries grasp information; if the >20 Hz bands carry *comparable or greater* information, that is the first sign the effect is myogenic and it should be reported at the next Figure Clinic, not at the end of the experiment.

**Controls.** Ours are three of the five in the E2 protocol:
- **Band-resolved decoding** — report per frequency band, always, never pooled.
- **Spatial specificity** — recompute with peripheral, temporal and frontal channels ablated.
- **Myogenic nuisance regressor** — supply the component so Track C can report `I(grasp ; EEG | EMG, myo)` alongside `I(grasp ; EEG | EMG)`. **The difference between those two numbers is the honesty figure of the entire project.**

The remaining two controls — a degradation manipulation that also perturbs the artifact, and a discriminating null — are Track C's, and are specified in the joint pre-registration.

**Failure modes.**
- *ICA over-correction removing movement-correlated neural activity along with ocular artifact.* Detect: decoding drops sharply after ICA on central channels. Recover: validate the ICA step against a planted effect exactly as Track E validates surrogates; if it kills the planted effect, fix or drop it.
- *Onset labels misaligned.* Detect: anticipation appears at implausibly long leads. Recover: hand-verify 30 trials; this is why step 2 exists.
- *Inheriting the synchronisation assumption.* Detect: nobody in the track can say what the measured synchronisation error is. Recover: measure it.
- *Reporting a pooled CMI-relevant number.* Detect: any figure with one accuracy value and no band or montage label. Recover: re-run separated.

**Evaluation metrics.** Per-band decoding accuracy; full-versus-central-only accuracy; before/after myogenic-conditioning decoding accuracy. These are our diagnostics; Track C converts them into conditional MI.

**Statistical tests.** Paired Wilcoxon signed-rank across subjects for band comparisons and for full-versus-central-only. Bootstrap percentile CIs on every reported value. Permutation nulls from `eval/surrogates/`. Holm correction across the band family, stated in the pre-registration.

**Expected figures.**
- *Figure E2-A.1:* decoding accuracy by frequency band, per subject, with the null band.
- *Figure E2-A.2:* full montage versus central-only, paired per subject.
- *Figure E2-A.3:* before versus after myogenic conditioning, paired per subject.

**Interpretation.** Effect concentrated in 0.3–3 Hz, surviving central-only montage, largely surviving myogenic conditioning → **cortical, and Gate B may proceed on it.** Effect concentrated above 20 Hz, or collapsing under central-only montage, or largely disappearing after conditioning → **myogenic, and this is the finding.** Report it as the headline; Gate B is decided accordingly; the track has done its job.

**Decision gate.** **Gate B.** Track C decides; Track A supplies the evidence that makes the decision valid.

**Publication relevance.** Publication 2. Figure E2-A.3 is the honesty figure of the project and belongs in the main text regardless of which way it goes.

**Estimated duration.** 4 weeks.

**Complete when.** All three figures exist; Track C has integrated the controls; the joint pre-registration is signed.

---

## 5.3 E3-A — The anticipatory detector

**Objective.** Establish the real anticipatory operating curve for grasp onset and, where the data permits, grasp type.

**Scientific motivation.** H2 holds that a decodable fraction of EEG grasp information is available before EMG onset. This is EEG's structural advantage: `g_EEG` taps the state upstream of `g_EMG`, so before onset there is no EMG signal to be redundant with. The quantity a designer needs is not a single accuracy but the trade-off surface between lead time, true-positive rate and false positives per minute — and for grasp *type* that surface has not been published cleanly.

**Critically: E3 runs regardless of Gate B's outcome.** A Tier-1 null measured in post-onset windows says nothing about anticipation, because in the anticipatory regime there is no EMG signal to be redundant with. Gate B cannot terminate this experiment.

**Inputs.** WAY-EEG-GAL, epoched to EMG threshold-onset. MRCP features.

**Outputs.** Detection latency relative to onset; TPR; FP/min — **swept across operating points, not reported at one**.

**Pipeline.**
1. Onset labels from EMG threshold-crossing, hand-verified on ≥30 trials.
2. MRCP features in sliding windows preceding onset.
3. sLDA and EEGNet detectors, equal tuning budgets, nested inner-fold selection.
4. Threshold sweep across the full decision-value range.
5. At each threshold: mean detection lead, TPR, FP/min computed over rest periods.
6. Per-subject curves; bootstrap CIs.

**Expected intermediate results.** After step 3, detector decision values should show a visible pre-onset rise on averaged trials for the best subjects. If no subject shows this, stop and check onset alignment before sweeping.

**Controls.**
- **Time-reversed surrogate** — reverse the epoch; genuine anticipation should not survive.
- **Shuffled-onset null** — randomise onset times within the recording; detection should fall to the false-positive floor.
- **Pre-baseline-shift check** — confirm the detector is not keying on a baseline drift artifact by re-running with baseline correction applied over a window that excludes the pre-onset period.

**Failure modes.**
- *Detecting the cue rather than the movement.* Detect: leads cluster suspiciously near the cue-to-onset interval. Recover: epoch to onset, verify cue timing is not in the feature window.
- *Baseline drift masquerading as MRCP.* Detect: effect survives time reversal. Recover: the pre-baseline-shift check.
- *Reporting a single operating point.* Detect: one number in the figure. Recover: sweep.

**Evaluation metrics.** Lead time (ms before EMG onset); TPR; FP/min; per-subject spread.

**Statistical tests.** Permutation test against the shuffled-onset null at each operating point. Bootstrap CIs on the curve. Paired Wilcoxon across subjects for detector-family comparison. Holm correction across operating points, stated in the pre-registration.

**Expected figures.**
- *Figure E3-A.1:* lead time versus TPR at fixed FP/min, per-subject curves plus the group median.
- *Figure E3-A.2:* the same, for grasp type rather than onset, **where the grasp vocabulary supports it** — and a stated limitation where it does not.
- *Figure E3-A.3:* surrogate panel — the curve under time reversal and shuffled onset.

**Interpretation.** Usable lead at acceptable FP/min → anticipation becomes a co-primary contribution and Track C measures what it buys downstream. Detection only at unusable FP/min → **report the curve as the finding.** It is publishable either way, because the trade-off surface for grasp type has not been published cleanly and a designer needs it regardless of whether it is favourable.

**Decision gate.** **Gate C.** Requires both our curve and Track C's forward-feed latency measurement.

**Publication relevance.** Publication 3, in either direction.

**Estimated duration.** 3 weeks.

**Complete when.** Curve produced with per-subject spread and CIs; surrogate panel passing; handed to Track C for forward-feed.

---

# 6. DATASETS

**Standing rule: no dataset enters this project on the strength of a description.** Every corpus is downloaded, opened, recounted and verified against its publication before any modelling. Output is a dataset card in `data/`, co-reviewed with Track E. The card records the **exact URL used and the access date** — never a remembered one.

---

## 6.1 WAY-EEG-GAL

**Purpose.** The only public corpus with synchronized EEG + EMG + kinematics + force and labelled behavioural events including movement onset. **Without it there is no E2 and no E3.**

**Download source.** Published as a data descriptor in *Scientific Data* (Luciw, Jarocka & Edin) with the corpus hosted on figshare. Locate via the data descriptor's own accession link; record the resolved URL, version and access date in the dataset card. Do not use a mirror without recording that it is one.

**Expected preprocessing.**
- Band-pass 0.3–3 Hz for the MRCP path; 8–30 Hz for the band-power comparison path.
- Surface Laplacian.
- ICA for ocular artifact — **validated against a planted effect before adoption**.
- Epoch relative to **EMG threshold-onset**. Pre-onset and post-onset windows kept separate throughout, never pooled.
- Per-channel normalisation fitted **within fold**.

**Splits.** Session-wise where the recording structure permits; otherwise trial-block. **Never window-level.** Splits come from `eval/splits/` and are pre-registered before any modelling result is reported.

**Known issues.** Synchronisation between modalities must be *measured*; multimodal corpora have documented synchronisation error and it is a known failure mode. Event labels must be inspected rather than trusted.

**Biases.** All able-bodied. All young-adult laboratory participants. Single laboratory, single protocol.

**Limitations.** n = 12. Narrow grasp vocabulary. Degradation conditions available to us are *simulated*, not physiological.

**Common mistakes.**
1. **Using it for grasp-*type* diversity claims.** It does not support them. This error has already been made once in this project's history and was caught by cross-document review.
2. **Trusting synchronisation instead of measuring it.**
3. **Epoching to cue rather than onset**, which silently converts an anticipation result into a cue-response result.
4. Pooling pre- and post-onset windows, which conflates a bandwidth claim with a temporal one.

**Recommended sanity checks.**
- Recount trials and subjects against the data descriptor.
- Verify channel count, sampling rate and channel names against the descriptor.
- Measure EEG–EMG synchronisation error and record the number.
- Hand-verify EMG threshold-onset on ≥30 randomly chosen trials.
- Confirm event-code semantics against the descriptor's own table.

**Required visualisations before modelling.**
- Raw traces for 10 random epochs, EEG and EMG on a common time axis, with the detected onset marked.
- Per-channel power spectral density, averaged and per subject, to identify bad channels and line noise.
- ERD/ERS time-frequency map over sensorimotor channels, onset-aligned — **you should be able to see the phenomenon before you try to decode it.**
- Grand-average MRCP, onset-aligned, per subject.
- Artifact-rejection rate per subject.

---

## 6.2 BCI-IV-2a (and 2b)

**Purpose.** Standard motor-imagery benchmark for decoder sanity-checking and for the T1-R reproduction. **Not an experimental dataset for Regenesis claims.**

**Download source.** BNCI Horizon 2020 dataset repository / the BCI Competition IV archive. Record the resolved URL and access date.

**Expected preprocessing.** Standard MI pipeline: band-pass 8–30 Hz, epoch per the source paper's timing (recorded explicitly), CSP or Riemannian tangent-space features.

**Splits.** Session-wise — the dataset provides distinct sessions and this is exactly what makes it useful for demonstrating our splitting policy.

**Known issues.** Class definitions and epoch timing vary between published analyses of the same data; this is the main source of reproduction deltas.

**Biases.** MI paradigm, cued, laboratory setting. Different task structure from grasping.

**Limitations.** It is motor imagery, not grasp. Nothing about it bears on H1.

**Common mistakes.**
1. **Reporting a benchmark number as if it bore on H1.** It establishes that the pipeline works. Nothing more.
2. Comparing our value to a published one without extracting the published epoch timing and band definition first.

**Recommended sanity checks.** Verify subject count, session structure, class labels and trial counts against the competition documentation.

**Required visualisations before modelling.**
- Per-class ERD time-frequency maps over C3/C4 — the effect should be visible.
- First two CSP or tangent-space components, coloured by class.
- Per-subject class balance.

---

## 6.3 EEGMMIDB

**Purpose.** Secondary MI benchmark, available if T1-R's primary target proves unsuitable. Optional.

**Download source.** PhysioNet. Record the resolved URL and access date.

**Expected preprocessing, splits, checks.** As BCI-IV-2a.

**Known issues.** Documented annotation irregularities for a subset of subjects — search the errata before use and record what you find in the dataset card.

**Common mistakes.** Using it without checking the errata.

---

# 7. LITERATURE

**Standing rule.** Predict-then-read: write expected method, expected result and one expected limitation **before** opening the paper. Grade A–F on first read (§0.2), extract *n*, population, task, class count, chance level and offline/online, and enter in the Evidence Table. **A paper is presented at Journal Club only after one of its numbers has been reproduced or a documented attempt has failed.**

Volume is not the objective. Roughly six papers read to the depth of being able to redraw their figures from memory beats forty skimmed.

---

## 7.1 Must read

**Pfurtscheller & Lopes da Silva — event-related desynchronisation and synchronisation.**
*Why we read it:* the phenomenon the entire EEG path rests on. *Insight it contributes:* what ERD is, its time course, its spatial distribution, and how much it varies across people. *Experiment that depends on it:* all of them — you cannot interpret a band-power result without it. *Assumptions:* supports the premise that sensorimotor rhythms carry movement-related information; its variability findings challenge any expectation of a subject-general decoder.

**Muthukumaraswamy / Whitham — muscle artifact contamination of scalp EEG.**
*Why we read it:* **the single most important paper for this track.** It is the mechanism behind our most probable false positive. *Insight it contributes:* where myogenic activity dominates in frequency (broadly, above ~20 Hz) and in space (peripheral, temporal, frontal sites). *Experiment that depends on it:* E2-A's three controls are constructed directly from these two facts. *Assumptions:* challenges any uncontrolled positive result for H1; supports the claim that MRCP features are comparatively protected.

**Niazi et al. — MRCP-based movement detection.**
*Why we read it:* the feature family we adopted over band power. *Insight it contributes:* how MRCP detection is done and at what operating point — the reported true-positive rate and lead. *Experiment that depends on it:* E3-A's detector design. *Assumptions:* supports the MRCP-over-band-power decision recorded in the project's decision log.

**Lew et al. — anticipatory movement-intent detection.**
*Why we read it:* the evidence base for H2 and for the lead times we consider plausible. *Insight it contributes:* achievable pre-onset detection latency and its false-positive cost. *Experiment that depends on it:* E3-A. *Assumptions:* supports H2; the false-positive figures challenge any assumption that anticipation is free.

**Lawhern et al. — EEGNet.**
*Why we read it:* our compact-CNN decoder arm. *Insight it contributes:* why a small, structured architecture outperforms larger ones on small EEG datasets — the design reasoning matters more than the architecture. *Experiment that depends on it:* T1-R, E2-A, E3-A. *Assumptions:* supports the choice of three decoder families to separate method effect from signal effect.

**Lotte et al. — review of classification algorithms for EEG-based BCIs.**
*Why we read it:* prevents reinvention and over-reach. *Insight it contributes:* the method landscape and, importantly, why simple methods frequently win at our sample sizes. *Experiment that depends on it:* T1-R baseline selection. *Assumptions:* supports keeping a linear baseline in every comparison.

**Luciw, Jarocka & Edin — WAY-EEG-GAL data descriptor.**
*Why we read it:* our spine dataset. *Insight it contributes:* exactly what was recorded, how, what the event labels mean, and what the protocol was. *Experiment that depends on it:* E2-A, E3-A. *Assumptions:* its protocol description is the ground truth against which our dataset card is verified.

---

## 7.2 Strongly recommended

**Barachant / Congedo — Riemannian approaches to BCI.**
*Why:* our Riemannian tangent-space decoder arm and, later, cross-session alignment. *Insight:* why covariance-based representations are robust to some forms of non-stationarity. *Depends:* T1-R, E2-A, and the Week 13+ encoder for E4. *Assumptions:* supports the belief that session drift is partially correctable geometrically.

**Schwarz & Müller-Putz — reach-and-grasp decoding from EEG.**
*Why:* the closest published analogue to our task. *Insight:* what class cardinality is achievable and with which features — this is the source of the low-70s binary and mid-60s four-class figures in our belief table. *Depends:* RQ-A1, RQ-A2. *Assumptions:* supports the discrete-selector architecture; **its exact values are being re-verified by Track E's Numbers Audit and must not be used as a threshold until that clears.**

**Varoquaux — cross-validation pitfalls in neuroimaging.**
*Why:* calibrates how to read every number we compare ourselves against. *Insight:* why small-*n* cross-validation is unstable and how it misleads. *Depends:* every experiment's split policy. *Assumptions:* supports the block-wise splitting mandate.

---

## 7.3 Background

**Surrogate-data methodology.** *Why:* you use `eval/surrogates/` daily and must know what each surrogate preserves and destroys. *Insight:* the logic of constructing a null that removes the effect while preserving nuisance structure. *Depends:* every experiment's control panel.

**BCI illiteracy literature.** *Why:* the 15–30% constraint is a design parameter, not trivia. *Insight:* that system performance is partly a property of the user. *Depends:* RQ-A5; the EMG-only fallback in the architecture.

**Statistical power in neuroscience.** *Why:* at n = 12 a null and an underpowered study are easy to confuse. *Insight:* why underpowered positives are unreliable even when significant. *Depends:* how we report every null.

---

## 7.4 Historical

**Bernstein — the degrees-of-freedom problem.** *Why:* the origin of the dimensionality framing the whole project inherits. *Insight:* why the motor system must be simplifying a problem it cannot solve variable-by-variable. *Depends:* nothing directly in Track A; read for shared vocabulary with Track B.

**Gallego, Perich & Miller — neural manifolds.** *Why:* read specifically to understand **why the scalp analogy is a category error.** The invasive manifold literature rests on hundreds of simultaneously recorded single units; we have ~16–32 channels of volume-conducted scalp potential. *Insight:* what the manifold claim actually requires. *Depends:* nothing — **do not import its methods.** *Assumptions:* its requirements are what made "neural synergies from EEG band power" a rejected mechanism in this project.

**EEG foundation-model literature.** *Why:* exploratory only, so the track can say why it is not adopted in Phase II. *Depends:* nothing.

---

# 8. SOFTWARE

## 8.1 Where Track A's code lives

```
regenesis/
├── data/                    # loaders + version pointers + dataset cards
│   ├── way_eeg_gal.py
│   ├── bci_iv_2a.py
│   └── cards/               # one markdown card per corpus
├── preprocessing/           # ◄ TRACK A OWNS
│   ├── eeg_filters.py       # band-pass, notch, Laplacian
│   ├── eeg_artifact.py      # ICA, rejection; includes the planted-effect validator
│   ├── epoching.py          # onset-relative epoching
│   └── features_eeg.py      # MRCP + band power, band-resolved routing
├── eeg/                     # ◄ TRACK A OWNS
│   ├── decoders_linear.py   # sLDA, CSP
│   ├── decoders_riemann.py  # tangent-space
│   ├── decoders_eegnet.py
│   ├── montage.py           # full / central-only ablation utilities
│   ├── myogenic.py          # high-frequency myogenic component estimator
│   └── anticipation.py      # E3 detector + threshold sweep
├── eval/                    # ◄ TRACK E OWNS — CONSUME ONLY, NEVER COPY
├── experiments/
│   ├── t1r_reproduction.yaml
│   ├── e2a_controls.yaml
│   └── e3a_anticipation.yaml
├── configs/                 # shared config fragments; seeds pinned
├── prereg/                  # PR-YYYY-NN.md, immutable after sign-off
├── results/         ... (27 KB left)